# 第08课：RAG智能助手项目
本课把第6课的“带原文回答”和第7课的“有限工具循环”，接进第二阶段已有的FastAPI会话后端。不是再造一个孤立项目，也不是提前做Coding Agent。

本课只聚焦三件事：**选路径并复用 → 成功后原子保存 → 用固定问题检查质量**。先读每组的短解释，再动手；HTML只供回看，不要求一口气看完。

## 0. 一条问题的完整去向
```text
客户端POST question+mode
  → FastAPI检查请求体和会话是否存在
  → run_request（本课作业1）
       rag：retrieve → 第6课answer_question → 引用检查
       tool：第7课run_agent → 许可/配对/预算/重复
  → save_exchange（本课作业2）→ 教师save_pair同事务存两条
  → JSON响应：业务状态、答案、来源、调用次数、是否保存
  → evaluate_case（本课作业3）：对照固定预期，不更改答案
```
mode由客户端明确选择，不让模型猜路由，也不允许客户端指定白名单。RAG路径不调用工具模型；tool路径不检索原文。RAG也可以成为Agent工具，本课先保留两条可独立定位的问题路径。

**范围**：只读虚构实验中心数据；新增接口逐次独立处理问题，数据库保存历史不表示自动具备跨轮代词理解。旧聊天接口继续保留模拟回复，使用本课新路径才会调用助手。没有用户登录/多租户、并发数据库切换、任意Shell或生产部署；本地教学服务只绑定127.0.0.1。

真实运行与代码测试分开记录，见[真实项目演练记录](真实项目演练记录.md)。创建验证使用仓库外参考实现，不把答案写进作业，也不把教师运行视作你独立完成。

## 运行前：先装到正在使用的环境
在本课目录执行：
```sh
python -m venv .venv
```
macOS/Linux：
```sh
.venv/bin/python -m pip install -r requirements.txt
.venv/bin/python -m pytest -q test_project_support.py
```
Windows PowerShell：
```powershell
.venv/Scripts/python.exe -m pip install -r requirements.txt
.venv/Scripts/python.exe -m pytest -q test_project_support.py
```
python是所选解释器；-m pip使用它的包管理器；install安装；-r读取清单。fastapi是HTTP框架，uvicorn启动服务，starlette是底层Web组件（<1与httpx的TestClient组合验证）；httpx发送HTTP请求并支持测试；pydantic校验字段；openai连接DeepSeek兼容接口；python-dotenv只读本地配置；pytest做离线回归。没有新增Embedding模型或向量库。

Notebook要在**当前内核**安装：新格执行 `%pip install -r requirements.txt`，必要时重启，再运行准备格。另一个终端装过不代表本内核可用。本机已为本课建好.venv；没有改动旧课环境。公开.env.example只是占位，真实配置读本课.env，否则复用第7课.env；别把密钥贴进Notebook。

In [ ]:
from pathlib import Path
import sys
import json
from copy import deepcopy

_candidates = [Path.cwd(), Path.cwd() / "03_大模型应用开发" / "08_RAG智能助手项目"]
COURSE_DIR = next((p for p in _candidates if (p / "project_support.py").is_file()), None)
if COURSE_DIR is None:
    print("❌ 请在本课目录或仓库根目录启动Notebook。")
else:
    if str(COURSE_DIR) not in sys.path:
        sys.path.insert(0, str(COURSE_DIR))
    from project_support import (
        observe_rag, observe_tool, UNKNOWN_ANSWER,
        make_rag_messages, retrieve_lab,
    )
    print("✅ 准备完成；尚未请求模型，也未创建数据库。")

## 1. 先选路径：不是所有问题都走同一套流程
### 1.1 一个请求由谁提供什么？
所有字段名是开发者的协议约定，不是Python关键字。

| 位置/字段 | 类型 | 谁提供、含义 |
|---|---|---|
| URL conversation_id | int | 客户端引用此前创建的会话编号 |
| 请求 question | str，必填，1～500字符 | 客户端问题 |
| 请求 mode | "rag"/"tool" | 客户端选择；省略为rag，显式tool走工具，None不合法 |
| 响应 status/answer | str | 复用业务函数结果；HTTP200不保证业务成功 |
| 响应 sources | list[dict] | RAG从实际证据解析的id/source；工具路径固定空列表 |
| 响应 model_calls/tool_calls | int | 模型请求次数/execute_call处理请求数，后者含拒绝与失败 |
| 响应 saved | bool | 程序是否已提交成功消息对，不由客户端指定 |

第6课的候选hits是原文，不是答案；第7课的工具请求也不是工具结果。本课检索复用文字重合**真实基线**，不是Embedding；没有以手填向量冒充真实编码。
### 1.2 先在本地看清“发给模型的原文在哪”
make_rag_messages(question:str,evidence:list[dict])→list[dict]来自第6课学生实现。本格只构造消息，尚不联网；json.loads是标准库解析函数，这里把user正文读回字典观察。

In [ ]:
library_evidence = [
    {"id": "lib-p1", "source": "借阅规则#p1",
     "text": "云城图书馆每人最多借3本书，借期14天。"},
]
library_question = "云城图书馆最多借几本书，借多久？"
library_messages = make_rag_messages(library_question, library_evidence)
print("规则所在角色：", library_messages[0]["role"])
print("实际放进user的资料：", json.loads(library_messages[1]["content"]))
print("变化：空证据时，第6课answer_question直接返回no_evidence，不问模型。")

### 1.3 真实RAG观察：本格打开后会产生请求
observe_rag(question:str,evidence:list[dict])→dict是教师函数，复用第6课answer_question，真实调用DeepSeek，检查JSON/字段/引用ID。返回result/model/usage/trace；trace是教师保存的请求响应轨迹，不是SDK对象。它不调用本课TODO，也不保存数据库。空证据零模型请求；有证据至多一次，超时不自动重试。引用通过只证明ID来自本次证据，不自动证明原文支持所有事实。

开关默认False只是避免“全部运行”重复计费，不是用假响应替代真实实验。创建阶段已真实演练，完成后可自行打开再观察。

In [ ]:
RUN_REAL_RAG = False  # 改True才运行；有证据时一次真实请求
rag_observation = None
if not RUN_REAL_RAG:
    print("⏭️ 未重复请求；可读真实项目演练记录。")
else:
    try:
        rag_observation = observe_rag(library_question, library_evidence)
        print("模型：", rag_observation["model"])
        print("结果：", rag_observation["result"])
        print("usage：", rag_observation["usage"])
        print("请逐项对照原文，引用存在不等于事实已自动证明。")
    except Exception as _live_error:
        print("❌ 真实请求未完成：", type(_live_error).__name__, "；不自动重试，不打印密钥或异常全文。")

### 1.4 真实工具观察：先请求，程序执行，再回答
observe_tool(question:str)→dict也是教师函数，复用你第7课run_agent；给模型find_room/get_devices的只读工具说明。本次教师演示仍查询本课虚构机房，固定数据room-a有24台；不是实际学校系统。最多4次模型请求，tool_choice=auto，模型实际选择与停止结果都记录，不假装一定成功。返回result/model/usage/trace；result内部messages是轨迹，公开HTTP响应不直接泄露它。

In [ ]:
RUN_REAL_TOOL = False
tool_observation = None
if not RUN_REAL_TOOL:
    print("⏭️ 未重复请求；先看记录中的工具名称、参数和实际次数。")
else:
    try:
        tool_observation = observe_tool("请查询room-a有多少台电脑，必须使用get_devices，不要猜。")
        print("模型：", tool_observation["model"])
        print("状态与答案：", tool_observation["result"]["status"], tool_observation["result"]["answer"])
        print("模型/工具次数：", tool_observation["result"]["model_calls"], tool_observation["result"]["tool_calls"])
        for _turn in tool_observation["trace"]:
            print("本次模型返回：", _turn["reply"])
        print("usage：", tool_observation["usage"])
    except Exception as _live_error:
        print("❌ 真实工具观察未完成：", type(_live_error).__name__, "；不自动重试。")

### 课堂练习1：园区助手走哪条路径？
目标是分开两个数据来源，不写类型边界算法。只填route_choices，值为rag或tool。
a“园区预约规定写了什么？”需要原文引用；b“查询gate-east当前教学数据中的开放状态”需要只读查询；c“园区资料没写馆长出生地”仍属于资料问答，资料不足应拒答，不能偷偷改工具路径。预期三项分别保持指定业务来源。

In [ ]:
route_choices = {"a": "", "b": "", "c": ""}

In [ ]:
try:
    assert isinstance(route_choices, dict), "请填写route_choices字典。"
    for _check_key, _check_expected in [("a", "rag"), ("b", "tool"), ("c", "rag")]:
        assert route_choices.get(_check_key), f"先填写{_check_key}。"
        assert route_choices[_check_key] == _check_expected, f"{_check_key}：按资料问答或只读查询选择，不因资料不足擅自换路径。"
    print("✅ 课堂练习1：两条路径已区分。")
except AssertionError as _check_error:
    print("❌ 课堂练习1：", str(_check_error))
except (NameError, TypeError, KeyError):
    print("❌ 课堂练习1：先定义route_choices字典，a/b/c填rag或tool。")

<details><summary>完成后查看要点</summary>

a/c属于rag；b属于tool。模式不是执行授权。模型即使提出未知工具，仍由第7课执行器拦截。
</details>

## 2. 接进HTTP和会话：返回了结果，不一定应该保存
### 2.1 复用什么，新增什么？
project_api.py是教师HTTP接线。它通过project_support导入第二阶段chat_app的repository和会话router，不导入旧main的lifespan，因此不会创建或改写旧课chat.sqlite3。新服务启动时在本课创建assistant.sqlite3；数据库是本机运行数据，Git已忽略。

新create_app(db_path:Path|None=None, ports_factory=real_ports, retrieve=retrieve_lab)→FastAPI，默认本课数据库；测试传临时路径。ports_factory是每次请求创建回调集合的工厂函数，不是模型；真实对象的generate/ask分别供RAG/工具使用。None在db_path位置表示使用默认路径，不表示不保存。同步HTTP路由由FastAPI调度；本课单个应用实例，repository配置全局，不支持一个进程并发切换多数据库。

| HTTP接口 | 含义 |
|---|---|
| GET /health | 只检查服务进程，不证明DeepSeek可用 |
| POST /conversations，JSON {"title":"实验"} | 旧后端创建会话，201返回id |
| POST /assistant/conversations/{id}/messages | 新助手接口，200返回业务结果与saved |
| GET /conversations/{id}/messages | 查看已提交历史 |

旧POST /conversations/{id}/messages仍是原来的模拟回复接口，不要与新增/assistant路径混用。

AskBody/AnswerBody是教师定义的Pydantic模型类，继承BaseModel以获得字段校验和序列化，沿用第3课。FastAPI看到body:AskBody，先把HTTP JSON校验为对象；你的业务函数中没有手工json.loads请求体的对应行。它再依据response_model=AnswerBody过滤/校验响应字段；这不证明答案事实正确。

### 2.2 在真实服务器中操作
完成作业1/2之前，新助手端点返回501“先完成”，不是环境坏了。GET /health、创建会话等旧接口现在即可运行。创建阶段已用仓库外参考实现真实演练完整路径，未在学生文件预填答案。

终端A在本课目录运行（macOS；Windows把解释器改为.venv/Scripts/python.exe）：
```sh
.venv/bin/python -m uvicorn project_api:app --host 127.0.0.1 --port 8008
```
project_api是模块名，app是该模块的FastAPI对象；不是文件路径。浏览器打开 http://127.0.0.1:8008/docs ，先POST /conversations创建会话，记住id，再POST新助手端点：
```json
{"question":"实验预约须在什么时候提交？","mode":"rag"}
```
切换tool：
```json
{"question":"请用get_devices查room-a电脑数量，不要猜。","mode":"tool"}
```
HTTP200后的status仍可能是no_evidence/no_answer/invalid_citations/step_limit/repeated_call，不能只看状态码。mode省略为rag，显式None会422；会话不存在404且不请求模型；TODO501；模型/存储异常502且不自动重试，不把异常解释为资料不足。新增端点不接受客户端allowed_names，额外字段422。

[FastAPI请求/测试](https://fastapi.tiangolo.com/tutorial/testing/) · [响应模型](https://fastapi.tiangolo.com/tutorial/response-model/)

### 在Notebook验证框架自动行为（离线，不代替真实HTTP）
下面TestClient来自fastapi.testclient，在进程内发请求，不开启TCP端口、不调用模型。真正的TCP＋DeepSeek记录已另行验证。TemporaryDirectory来自标准库tempfile，是退出with后清理的临时目录；只在这里放测试数据库。create_app是教师工厂；offline_ports是教师上下文管理器，提供禁止真实调用的回调。

Client.post(path,json=dict)把字典编码成HTTP JSON；response.status_code是HTTP状态码，response.json()解析返回正文为Python对象。使用with会执行应用启动/关闭流程，启动时初始化临时数据库。以下缺question的请求在进入业务函数前就422，错误转换由FastAPI声明触发，不是你的函数手写判断。


In [ ]:
from tempfile import TemporaryDirectory
from fastapi.testclient import TestClient
from project_api import create_app
from project_support import offline_ports

with TemporaryDirectory() as _demo_dir:
    with TestClient(create_app(Path(_demo_dir) / "demo.sqlite3", offline_ports)) as _demo_client:
        _created = _demo_client.post("/conversations", json={"title": "临时演示"})
        _cid = _created.json()["id"]
        print("创建会话：", _created.status_code, "；id：", _cid)
        _invalid = _demo_client.post(f"/assistant/conversations/{_cid}/messages", json={"mode": "rag"})
        print("缺少question：", _invalid.status_code, "；业务尚未执行。")
        _missing = _demo_client.post("/assistant/conversations/999/messages", json={"question": "问题"})
        print("会话不存在：", _missing.status_code, "；未请求模型。")
        print("历史仍为空：", _demo_client.get(f"/conversations/{_cid}/messages").json())


### 2.3 什么叫“成功提交一对消息”？
本课只把status为answered或complete且answer非空白的结果，保存成一条user＋一条assistant。no_answer虽然可能有“资料不足”的正文，但不算成功历史；预算/重复/引用错误也不保存。HTTP响应照常展示状态，不保存不等于不给用户提示。

原子提交：两条消息使用同一个数据库事务，全部成功才提交；第二条写入失败时第一条也回滚。教师store_pair(db_path, conversation_id, question, answer)负责SQL与事务，学生只通过save_pair(conversation_id, question, answer)回调一次。不是先存user，再赌模型成功。

这不是“模型请求”和“工具结果”的消息配对：数据库旧表只保存成功的user/assistant正文，内部assistant/tool轨迹只用于本次处理与审计，不往旧表塞role=tool。本课下一次模型调用不自动读取数据库历史，因此跨轮追问不在本课承诺范围。

【先预测，再运行】某次HTTP200，status=no_answer，answer="资料不足，无法回答。"：是否应该新增成功历史？用bool填写prediction_save。

In [ ]:
prediction_save = None

In [ ]:
try:
    assert type(prediction_save) is bool, "请填写True或False。"
    assert prediction_save is False, "no_answer不是成功历史，即使HTTP200且有文字。"
    print("✅ 预测通过：HTTP成功与成功提交分开。")
except (NameError, AssertionError):
    print("❌ 预测：请用bool判断no_answer是否应写入成功历史。")

### 课堂练习2：园区会话要不要保存？
只填save_choices的bool值。a answered且正文"请提前预约"；b no_answer且正文"资料不足"；c complete但正文全空格；d step_limit且answer为空。目标：只提交完整非空的成功业务结果，不写数据库。

In [ ]:
save_choices = {"a": None, "b": None, "c": None, "d": None}

In [ ]:
try:
    assert isinstance(save_choices, dict), "请填写字典。"
    for _check_key, _check_expected in [("a", True), ("b", False), ("c", False), ("d", False)]:
        assert type(save_choices.get(_check_key)) is bool, f"{_check_key}需填写bool。"
        assert save_choices[_check_key] is _check_expected, f"{_check_key}：检查业务状态及正文是否非空，不能只看有无文字。"
    print("✅ 课堂练习2：成功提交条件通过。")
except AssertionError as _check_error:
    print("❌ 课堂练习2：", str(_check_error))
except (NameError, TypeError):
    print("❌ 课堂练习2：先定义save_choices字典，a/b/c/d均需bool。")

<details><summary>完成后查看要点</summary>

只有a保存。b不是成功业务结果；c没有有效正文；d没有最终回答。自动检查不验证真实SQL，原子提交另有教师事务测试。
</details>

## 3. 测试通过，与真实回答正确是两张成绩单
### 3.1 代码回归和固定问答分别测什么？
离线测试用假模型让错误可复现，检查路径隔离、引用回溯、未知工具拦截、消息提交与评估规则。它们不能证明真实模型会选正确工具或保留原文限制。

真实固定评估则冻结问题、预期、版本和预算，再通过实际HTTP＋模型运行，不改题迎合结果。evaluation_cases.json含6例；模型输出波动可导致失败，保留原样，不反复挑最好一次。无依据例也是有效评估，但不要混入“相关资料命中率”的分母。本课只报告固定问题通过数/6，不宣称普遍准确率。

教师演示图书馆：原文“周一闭馆、周二至周日开放”，预期回答须包含“周一”，来源必须lib-p1。回答“每天开放”即使格式、引用ID都正确，固定关键文字检查仍失败；反之包含“周一”也未必语义正确，仍需人核对原文，不能把关键词匹配称作事实证明。

In [ ]:
evaluation_example = {
    "id": "library-closed",
    "expected_status": "answered",
    "must_include": ["周一"],
    "expected_source_ids": ["lib-p1"],
    "max_model_calls": 1,
}
observed_example = {
    "status": "answered", "answer": "周一闭馆。",
    "sources": [{"id": "lib-p1", "source": "开放规则#p1"}],
    "model_calls": 1, "tool_calls": 0,
}
print("冻结预期：", evaluation_example)
print("本次结果：", observed_example)
print("反例：answer改成每天开放，失败类别为answer；来源改成lib-other，失败类别为sources。")
print("变化：来源顺序不作为质量目标，本课比较id集合，不比较文件路径的拼写。")

### 3.2 本课评估字段和失败类别
case/result均是普通dict，不是SDK对象。case由开发者冻结；result来自本次公开助手结果，不从模型口头“我成功了”取结论。

| case字段 | 类型与含义 |
|---|---|
| id | str，固定评估样例编号，不是工具调用ID |
| expected_status | str，应返回的业务状态 |
| must_include | list[str]，正文必须包含的全部片段；空列表不约束正文 |
| expected_source_ids | list[str]，本次来源id集合必须与之相等；空列表要求无来源 |
| max_model_calls | 非负int，模型调用数上限；0表示不能问模型，不是无限 |

evaluate_case返回case_id/passed/failures。failures是list[str]，顺序固定：status（状态不符）、answer（缺少任一片段）、sources（来源集合不符）、budget（模型调用数超出上限）；全部满足则passed=True、failures=[]。多个错误要全部报告，不只返回第一项。它不请求模型、不改case或result，也不负责重新运行样例。

记录时区分请求次数与token；usage缺失时用量未知，不把估算写成实测账单。程序失败是独立失败，不能改写成no_answer来让评估通过。
### 课堂练习3：园区结果是哪一关失败？
仅填audit_slots，值用answer/sources/budget。三例的状态均正确，其他未提及项均正确：
a预期正文含“18:00”，实际仅“可随时取消”；b预期source ID为gate-p1，实际gate-p2；c模型预算最多1次，实际2次。

In [ ]:
audit_slots = {"a": "", "b": "", "c": ""}

In [ ]:
try:
    assert isinstance(audit_slots, dict), "请填写字典。"
    for _check_key, _check_expected in [("a", "answer"), ("b", "sources"), ("c", "budget")]:
        assert audit_slots.get(_check_key), f"先填写{_check_key}。"
        assert audit_slots[_check_key] == _check_expected, f"{_check_key}：区分缺关键文字、错来源、超出模型调用预算。"
    print("✅ 课堂练习3：失败原因已分开。")
except AssertionError as _check_error:
    print("❌ 课堂练习3：", str(_check_error))
except (NameError, TypeError):
    print("❌ 课堂练习3：先定义audit_slots字典，a/b/c填写失败类别。")

<details><summary>完成后查看要点</summary>

a是answer，b是sources，c是budget。这只是冻结规则检查，不自动证明语义。
</details>

### 3.3 作业完成后跑真实整套评估
终端A保持服务运行；终端B在本课目录执行：
```sh
.venv/bin/python run_evaluation.py --live
```
不加--live只打印说明，不发送HTTP。--base-url默认http://127.0.0.1:8008，可显式指定其他本机端口，None不是命令输入。本脚本只允许localhost/127.0.0.1。httpx.Client是第三方HTTP客户端，base_url是请求前缀、timeout=180是客户端等待设置、trust_env=False不读取代理配置；本地HTTP请求可能触发服务器去调用DeepSeek。它不是模型SDK。

脚本每例新建会话，顺序发6次助手HTTP请求；一例tool可能触发多次模型调用。每次重跑都会再次请求，不自动重试。若501，先补作业；502检查配置/网络/额度，保留失败记录。先用离线回归定位代码，再做真实观察，不能为得到6/6擅自放宽标准。

[真实演练结果与限制](真实项目演练记录.md) · [OpenAI官方工具调用分工](https://developers.openai.com/api/docs/guides/function-calling) · [DeepSeek接口](https://api-docs.deepseek.com/api/create-chat-completion/)

## 4. 课后作业：实验中心助手
只修改assistant_project.py三个TODO，不修改教师函数、测试、旧课答案或冻结样例。前文图书馆示例、园区课堂题与此处实验中心对象分开。28项作业测试对应“路径编排/提交/评估”，教师另12项，不重写数学算法或通用类型守卫。输入形状由HTTP/教师适配保证，模式仅rag/tool，不要求处理其他模式。

### 4.1 run_request：选择路径并统一公开结果
参数：
- question:str，用户原问题；mode:str，客户端已校验为rag或tool。
- retrieve:Callable，接收question，返回候选list[dict]，每项含id/text/source，可带score；真实接线为retrieve_lab。
- generate:Callable，接收第6课messages:list[dict]，返回已校验的status/answer/citations字典；不是SDK客户端。
- ask:Callable，接收第7课消息快照，返回规范assistant字典；不是观察函数observe_tool。
- registry:dict[str,Callable]、allowed_names:set[str]，沿用第7课，不由客户端传入。
- max_steps:int=4，仅传给工具循环的调用预算；省略4、显式0合法、None不合法，不增加通用判断题。

RAG路径只检索一次，复用已导入的第6课answer_question。零候选零generate；有候选一次generate，保留no_answer/invalid_citations等结果。工具路径只复用已导入的第7课run_agent，不调用retrieve/generate，保留循环status/answer和实际计数。

返回一个新dict，恰好mode/status/answer/sources/model_calls/tool_calls六键。例如：
```python
{"mode":"rag","status":"answered","answer":"17:00前提交。",
 "sources":[{"id":"rules-p1","source":"rules.md#p1"}],"model_calls":1,"tool_calls":0}
```
tool路径sources=[]，内部messages不公开；RAG路径model_calls为实际generate次数0或1、tool_calls=0。模型/校验异常继续抛出，不伪装不知道、不重试。不要创建客户端或读密钥。

### 4.2 save_exchange：只提交成功消息对
conversation_id:int，已存在会话ID；question:str原问题；result:dict为4.1结果，关键字段status/answer；save_pair:Callable接收三个位置参数(conversation_id, question, answer)，成功返回None，失败抛异常，自身保证两条消息同事务。
返回bool，表示是否完成提交。answered/complete且正文去空白后非空才调用save_pair恰好一次，保存的是原question与原answer，不修改正文。其他状态或纯空白正文不调用，返回False。异常自然抛出，不返回True。不得修改result。不在此函数自己写SQL或分两次调用保存。

### 4.3 evaluate_case：报告固定样例差异
case和result均dict；case字段见3.2，实际运行用evaluation_cases.json；result是4.1的六键公开结果。不要要求额外的case.mode/question参与检查，它们供运行脚本使用，不是评估条件。
输出例：
```python
{"case_id":"e1","passed":False,"failures":["answer","sources"]}
```
完整支持status、全部must_include片段、来源ID集合相等、model_calls不超max_model_calls四项；failures按status/answer/sources/budget顺序，允许同时多项；sources顺序/重复不影响集合比较。空must_include自动满足；空expected_source_ids要求结果无来源。只检查，不修改输入，不额外调用模型，不从关键词通过推断事实正确。

### 4.4 命令与验收边界
```sh
.venv/bin/python -m pytest -q test_assistant_project.py
.venv/bin/python -m pytest -q test_project_support.py
```
初始28项学生测试因NotImplementedError失败，12项教师测试应通过。完成作业1/2后真实服务器可处理新端点；完成作业3后再运行整套--live评估。创建验证参考实现放仓库外，三个学生函数保持空白。真实质量结果与代码测试分开报告；完成本课不等于生产安全或Coding Agent综合验收。

## 5. 理解题：先凭记忆回答，再检测
1. 本课业务路径由谁明确选择？
   - A：模型自动决定接口的路径
   - B：客户端明确选择业务路径
   - C：检索分数决定所有的权限
   - D：工具名称决定会话的编号

2. 回答引用ID均存在，能得出什么？
   - A：接口成功表示回答全正确
   - B：字段合法表示引用都真实
   - C：引用存在仍需核对原文义
   - D：正文非空表示资料已充分

3. 为何由save_pair一次提交两条消息？
   - A：两条消息必须同事务提交
   - B：先存问题再忽略回答失败
   - C：工具失败仍当成功来保存
   - D：模型返回即可写入成功史

4. 真实模型请求超时，应怎样看待？
   - A：请求失败统一标为无资料
   - B：模型超时自动再请求一次
   - C：接口报错仍写入成功历史
   - D：服务异常不能伪装资料少

5. 为何同时做离线测试和真实固定评估？
   - A：测试全过表示模型全答对
   - B：分别报告代码与回答质量
   - C：演示成功便不必固定评估
   - D：多次调用只保留最好结果

In [ ]:
quiz_answers = {"q1": "", "q2": "", "q3": "", "q4": "", "q5": ""}

In [ ]:
try:
    _check_keys = ["q1", "q2", "q3", "q4", "q5"]
    _check_correct = ["B", "C", "A", "D", "B"]
    _check_explanations = [
        "mode由客户端明确选择；省略默认rag，不由模型猜路径。",
        "引用ID存在只证明来自本次证据，仍需核对原文事实。",
        "同事务提交确保回答保存失败时不会留下半对成功历史。",
        "模型超时是服务失败，不是资料不足，不自动重试。",
        "离线测试验证代码契约，真实固定评估检查本次回答质量，不能互相替代。",
    ]
    assert isinstance(quiz_answers, dict), "请填写字典。"
    _check_failed = 0
    for _check_key, _check_expected, _check_explanation in zip(_check_keys, _check_correct, _check_explanations):
        _check_answer = quiz_answers.get(_check_key, "")
        if _check_answer not in ("A", "B", "C", "D"):
            print(f"❌ {_check_key}：先填写A/B/C/D。")
            _check_failed += 1
        elif _check_answer != _check_expected:
            print(f"❌ {_check_key}：{_check_explanation}")
            _check_failed += 1
    if not _check_failed:
        print("✅ 5道理解题全部通过。")
except (NameError, AssertionError, TypeError):
    print("❌ 请先运行quiz_answers答题格，按q1至q5填字母。")

## 6. 第二天闭卷复现
第二天另建草稿，复制三个函数的**空函数头**，清空草稿答案，保留正式作业和Notebook原样。先从记忆画出HTTP→路径→调用→保存→评估，再独立写一个路径和提交条件；最后用原测试定位缺口。

能解释“HTTP200但saved=False”“引用合法但关键事实错”“工具失败不能当无记录”三个反例，再决定是否进入第09课。不要因看过演练就记录为独立掌握。速查见[速查_项目闭环](速查_项目闭环.md)，需要继续讲解时直接发小节编号。